# CNN Regression -- Pepper CC vs NOCC Group Comparison (Corrected Groups, DNG Visualization)

Merges the CC-only group comparison (`Pepper_CNN_CC_GroupComparison.ipynb`) and the NOCC-only group
comparison (`Pepper_CNN_CC_3exp.ipynb`) into a single notebook that trains and compares **CC (Color
Corrected) vs NOCC (No Color Correction)** images for each of 3 experiment groups.

**Groups (`Comparison_Portakabin` CSVS00 removed from B & C):**
- **Group A**: `Comparison_on_Portakabin` `CSVS00` + `CSVS01` (unchanged)
- **Group B**: `Evaluation_on_Greenhouse/CSVS00` + `Evaluation_on_Portakabin/CSVS00`
  (`Comparison_Portakabin` CSVS00 removed)
- **Group C**: Group B + `Comparison_on_Portakabin/CSVS01`
  (`Comparison_Portakabin` CSVS00 removed)

**Key changes vs the two source notebooks:**
- Both **CC and NOCC** are trained for every group (5-fold CV each) and compared right after each group.
- **Class imbalance**: inverse-frequency `WeightedRandomSampler` on every training fold + target-noise
  augmentation, on top of the existing balanced per-source/per-class fold assignment (guarantees every
  data source appears in validation across folds).
- **Visualizations** (scatter, frieze) now use the **original raw DNGs**, center-cropped to the produce
  skin and matched to each TIFF by filename stem, instead of the processed JPEGs.
- **Frieze**: one row per score, capped at **10 images/row**.
- **Monitoring**: LR scheduler, early stopping and best-checkpoint selection are centered on **RMSE**
  (primary) and **R2** (secondary) rather than raw validation loss, while all standard metrics
  (MSE, RMSE, MAE, R2, Spearman rho) are still computed and reported.
- **Architecture**: `RegressionCNN` (depth=16, 6 conv blocks), `IMG_SIZE=64`, unchanged from the
  reference `260305_cnn_regression_analysis.ipynb`. Early-stopping patience=30, LR-scheduler patience=8.

## 1. Import Libraries

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from glob import glob
from collections import defaultdict, Counter
import copy
from skimage import io
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                              confusion_matrix, classification_report)
from scipy.stats import spearmanr, wilcoxon
import warnings
warnings.filterwarnings('ignore')

try:
    import rawpy
except ImportError as e:
    raise ImportError('rawpy is required to read DNG files. Install it with: pip install rawpy') from e
try:
    import cv2
except ImportError as e:
    raise ImportError('opencv-python is required for DNG resizing. Install it with: pip install opencv-python') from e

%matplotlib inline
mpl.rcParams['figure.figsize'] = 16, 8
plt.rcParams['font.size'] = 10

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Configuration

In [ ]:
# -- Dataset root --
PEPPER_ROOT = r'F:\Aymen\server\Database_CSVS\CSVS_Pepper_202602_ElEjido'

PEPPER_FOLDERS = {
    'Comparison_Portakabin': 'Pepper_COLOR_Comparison_on_Portakabin',
    'Evaluation_Greenhouse':  'Pepper_COLOR_Evaluation_on_Greenhouse',
    'Evaluation_Portakabin':  'Pepper_COLOR_Evaluation_on_Portakabin',
}

# -- Image / model hyper-params (identical to the 260305 reference notebook) --
IMG_SIZE   = 64      # px
DEPTH      = 16      # base CNN depth
BATCH_SIZE = 32
NUM_EPOCHS = 1000
LEARNING_RATE = 0.001

# -- Scheduler & early stopping --
LR_PATIENCE              = 8     # ReduceLROnPlateau patience
LR_FACTOR                = 0.5
EARLY_STOPPING_PATIENCE  = 30    # patience on the monitored metric (RMSE)

# -- Cross-validation --
N_FOLDS = 5

# -- DNG visualization --
DNG_CROP_FRACTION = 0.4   # central fraction of the DNG frame kept before resize (produce skin only)

print(f'PEPPER_ROOT : {PEPPER_ROOT}')
print(f'IMG_SIZE={IMG_SIZE}  DEPTH={DEPTH}  BATCH={BATCH_SIZE}  LR={LEARNING_RATE}')
print(f'LR patience={LR_PATIENCE}  ES patience={EARLY_STOPPING_PATIENCE}  N_FOLDS={N_FOLDS}')
print(f'DNG crop fraction={DNG_CROP_FRACTION}')

## 3. Data Loading -- CC & NOCC TIFFs per Outil + DNG Index

DNGs are indexed once per experiment root (excluding `OUTPUT`) and matched to each TIFF by filename
stem for visualization only. Training still uses the 16-bit `OUTPUT/<mode>/<outil>/Score=XX/*.tif`.

In [ ]:
def load_mode_data(experiment_root, mode, outil, source_key):
    """Load TIFFs for one outil from OUTPUT/<mode>/<outil>/Score=XX/*.tif.
    Returns a list of (tif_path, score_float, source_key)."""
    mode_dir = os.path.join(experiment_root, 'OUTPUT', mode, outil)
    if not os.path.isdir(mode_dir):
        print(f'  WARNING: {mode_dir} not found')
        return []

    data = []
    score_dirs = sorted(
        d for d in os.listdir(mode_dir)
        if os.path.isdir(os.path.join(mode_dir, d)) and d.startswith('Score=')
    )
    for sdir in score_dirs:
        score = int(sdir.split('=')[1])
        tifs = glob(os.path.join(mode_dir, sdir, '*.tif'))
        for tif in tifs:
            data.append((tif, float(score), source_key))
        if tifs:
            print(f'      [{mode}] {sdir}: {len(tifs)} images')
    return data


def build_dng_index(experiment_root):
    """Walk the experiment folder (skipping OUTPUT) and index .dng files by filename stem."""
    index = {}
    for dirpath, dirnames, filenames in os.walk(experiment_root):
        dirnames[:] = [d for d in dirnames if d != 'OUTPUT']
        for f in filenames:
            if f.lower().endswith('.dng'):
                stem = os.path.splitext(f)[0]
                index[stem] = os.path.join(dirpath, f)
    return index


# -- Load all experiments (CC + NOCC) and build the global DNG index --
all_data = {}    # all_data[key][outil]['cc' | 'nocc'] -> list of (tif, score, source_key)
dng_index = {}   # stem -> dng_path, merged across all experiment roots

for key, folder_name in PEPPER_FOLDERS.items():
    exp_root = os.path.join(PEPPER_ROOT, folder_name)
    print(f'\nIndexing DNGs for {key} ...')
    exp_dng_index = build_dng_index(exp_root)
    print(f'  Found {len(exp_dng_index)} DNG files')
    dng_index.update(exp_dng_index)

    all_data[key] = {}
    cc_root = os.path.join(exp_root, 'OUTPUT', 'cc')
    outils = sorted(os.listdir(cc_root)) if os.path.isdir(cc_root) else []
    for outil in outils:
        src_key = f'{key}/{outil}'
        print(f'  Loading {src_key} ...')
        all_data[key][outil] = {
            'cc':   load_mode_data(exp_root, 'cc',   outil, src_key),
            'nocc': load_mode_data(exp_root, 'nocc', outil, src_key),
        }
        print(f'    -> CC={len(all_data[key][outil]["cc"])}  NOCC={len(all_data[key][outil]["nocc"])}')

print(f'\nGlobal DNG index: {len(dng_index)} files')
print('Data loading complete.')

## 4. Group Definitions (A / B / C, CC & NOCC)

Groups B and C exclude `Comparison_Portakabin/CSVS00`.

In [ ]:
def build_groups(mode):
    """Group A/B/C for the given mode ('cc' or 'nocc').
    Groups B and C EXCLUDE Comparison_Portakabin/CSVS00."""
    group_A = (all_data['Comparison_Portakabin']['CSVS00'][mode] +
               all_data['Comparison_Portakabin']['CSVS01'][mode])
    group_B = (all_data['Evaluation_Greenhouse']['CSVS00'][mode] +
               all_data['Evaluation_Portakabin']['CSVS00'][mode])
    group_C = group_B + all_data['Comparison_Portakabin']['CSVS01'][mode]
    return {'Group_A': group_A, 'Group_B': group_B, 'Group_C': group_C}


GROUPS_CC   = build_groups('cc')
GROUPS_NOCC = build_groups('nocc')
GROUP_KEYS  = ['Group_A', 'Group_B', 'Group_C']
MODE_GROUPS = {'CC': GROUPS_CC, 'NOCC': GROUPS_NOCC}

GROUP_LABELS = {
    'Group_A': 'A: Comparison_Portakabin CSVS00+CSVS01',
    'Group_B': 'B: Greenhouse CSVS00 + Portakabin_Eval CSVS00 (Comparison CSVS00 removed)',
    'Group_C': 'C: Group B + Comparison CSVS01 (Comparison CSVS00 removed)',
}


def check_cc_nocc_parity(group_key):
    """Diagnostic: CC vs NOCC are expected to be the same physical captures processed
    two ways, so they should match 1:1. Flags any count/stem mismatch instead of
    assuming parity (used again in Section 10's paired significance test)."""
    cc_data = GROUPS_CC[group_key]
    nocc_data = GROUPS_NOCC[group_key]
    cc_stems = set(os.path.splitext(os.path.basename(t))[0] for t, _, _ in cc_data)
    nocc_stems = set(os.path.splitext(os.path.basename(t))[0] for t, _, _ in nocc_data)

    match_str = '[MATCH]' if len(cc_data) == len(nocc_data) else '[MISMATCH]'
    print(f'  CC/NOCC parity: CC N={len(cc_data)}  NOCC N={len(nocc_data)}  {match_str}')
    only_cc = sorted(cc_stems - nocc_stems)
    only_nocc = sorted(nocc_stems - cc_stems)
    if only_cc or only_nocc:
        print(f'    {len(only_cc)} stems only in CC, {len(only_nocc)} stems only in NOCC')
        if only_cc:
            print(f'      e.g. CC-only: {only_cc[:3]}')
        if only_nocc:
            print(f'      e.g. NOCC-only: {only_nocc[:3]}')
    else:
        print('    Stems fully match between CC and NOCC.')


def describe_group(group_key):
    print(f'\n{group_key} ({GROUP_LABELS[group_key]})')
    check_cc_nocc_parity(group_key)
    for mode, GROUPS in MODE_GROUPS.items():
        data = GROUPS[group_key]
        scores = sorted(set(int(s) for _, s, _ in data))
        sources = sorted(set(src for _, _, src in data))
        print(f'  [{mode}] N={len(data)}  scores={scores}')
        class_counts = Counter(int(s) for _, s, _ in data)
        print('      class counts: ' + ', '.join(f'{c}:{class_counts[c]}' for c in sorted(class_counts)))
        for src in sources:
            n = sum(1 for _, _, s in data if s == src)
            print(f'      source "{src}": {n} images  (~{n // N_FOLDS}/fold)')


for gk in GROUP_KEYS:
    describe_group(gk)

## 5. Dataset Class, Transforms & Class-Imbalance Sampler

In [ ]:
class PepperDataset(Dataset):
    """Loads 16-bit TIFF images for regression.
    data_list: list of (tif_path, score, source_key) -- source_key unused during training."""

    def __init__(self, data_list, img_size, transform=None, target_noise_std=0.0):
        self.data_list = data_list
        self.img_size = img_size
        self.transform = transform
        self.target_noise_std = target_noise_std

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        tif_path, score, _src = self.data_list[idx]
        try:
            arr = io.imread(tif_path)
            if arr.dtype != np.uint16:
                raise ValueError(f'Not 16-bit: {arr.dtype}')
            arr = arr.astype(np.float32) / 65535.0
            image = torch.from_numpy(arr.transpose(2, 0, 1)).float()
            if self.transform:
                image = self.transform(image)
            target = torch.tensor([score], dtype=torch.float32)
            if self.target_noise_std > 0:
                target = target + torch.randn_like(target) * self.target_noise_std
            return image, target
        except Exception as e:
            print(f'Error loading {tif_path}: {e}')
            return torch.zeros(3, self.img_size, self.img_size), torch.tensor([score], dtype=torch.float32)


# Transforms -- identical to the reference 260305 notebook (no RandomAffine)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomChoice([
        transforms.Lambda(lambda x: torch.rot90(x, k=0, dims=[1, 2])),
        transforms.Lambda(lambda x: torch.rot90(x, k=1, dims=[1, 2])),
        transforms.Lambda(lambda x: torch.rot90(x, k=2, dims=[1, 2])),
        transforms.Lambda(lambda x: torch.rot90(x, k=3, dims=[1, 2]))
    ]),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])


def make_weighted_sampler(data_list):
    """Inverse-class-frequency WeightedRandomSampler to counter score-class imbalance
    within a training fold (used in addition to target-noise augmentation and the
    balanced per-source/per-class fold assignment in Section 8)."""
    classes = [int(s) for _, s, _ in data_list]
    counts = Counter(classes)
    weights = np.array([1.0 / counts[c] for c in classes], dtype=np.float64)
    return WeightedRandomSampler(weights=torch.from_numpy(weights),
                                  num_samples=len(weights), replacement=True)


print('PepperDataset, transforms, and weighted sampler defined.')
print('Class imbalance handled via: (1) inverse-frequency WeightedRandomSampler on train folds,')
print('                             (2) target_noise_std augmentation,')
print('                             (3) balanced per-source/per-class fold assignment (Section 8).')

## 6. CNN Architecture

In [ ]:
class RegressionCNN(nn.Module):
    def __init__(self, input_channels=3, img_height=64, img_width=64, depth=16):
        super().__init__()
        self.conv_layers = nn.Sequential(
            # Block 1: depth
            nn.Conv2d(input_channels, depth,    3, padding=1), nn.BatchNorm2d(depth),    nn.ReLU(),
            nn.Conv2d(depth,    depth,          3, padding=1), nn.BatchNorm2d(depth),    nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 2: depth*2
            nn.Conv2d(depth,    depth*2,        3, padding=1), nn.BatchNorm2d(depth*2),  nn.ReLU(),
            nn.Conv2d(depth*2,  depth*2,        3, padding=1), nn.BatchNorm2d(depth*2),  nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 3: depth*4
            nn.Conv2d(depth*2,  depth*4,        3, padding=1), nn.BatchNorm2d(depth*4),  nn.ReLU(),
            nn.Conv2d(depth*4,  depth*4,        3, padding=1), nn.BatchNorm2d(depth*4),  nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 4: depth*8
            nn.Conv2d(depth*4,  depth*8,        3, padding=1), nn.BatchNorm2d(depth*8),  nn.ReLU(),
            nn.Conv2d(depth*8,  depth*8,        3, padding=1), nn.BatchNorm2d(depth*8),  nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 5: depth*16
            nn.Conv2d(depth*8,  depth*16,       3, padding=1), nn.BatchNorm2d(depth*16), nn.ReLU(),
            nn.Conv2d(depth*16, depth*16,       3, padding=1), nn.BatchNorm2d(depth*16), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Block 6: depth*32
            nn.Conv2d(depth*16, depth*32,       3, padding=1), nn.BatchNorm2d(depth*32), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.flattened_size = depth * img_height * img_width // 128
        self.fc_layers = nn.Sequential(
            nn.Linear(self.flattened_size,      self.flattened_size // 2), nn.ReLU(),
            nn.Linear(self.flattened_size // 2, self.flattened_size // 4), nn.ReLU(),
            nn.Linear(self.flattened_size // 4, 1), nn.ELU()
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        return self.fc_layers(x)


# Sanity check
_m = RegressionCNN(3, IMG_SIZE, IMG_SIZE, DEPTH)
_total = sum(p.numel() for p in _m.parameters())
print(f'RegressionCNN | flattened={DEPTH*IMG_SIZE*IMG_SIZE//128} | params={_total:,}')
del _m

## 7. DNG Central-Crop Visualization Helper

In [ ]:
_dng_crop_cache = {}
_dng_warned_missing = set()


def get_cropped_dng(tif_path, size=IMG_SIZE, crop_fraction=DNG_CROP_FRACTION):
    """
    Fetches the DNG matching a TIFF filename stem, demosaics it with rawpy, center-crops
    the central `crop_fraction` of the frame (produce skin, discarding background) and
    resizes to `size`x`size`. Returns a uint8 HxWx3 array, or None if the DNG can't be
    found or loaded (frieze cells fall back to an 'N/A' placeholder in that case).
    """
    stem = os.path.splitext(os.path.basename(tif_path))[0]
    if stem in _dng_crop_cache:
        return _dng_crop_cache[stem]

    dng_path = dng_index.get(stem)
    if dng_path is None:
        if stem not in _dng_warned_missing:
            print(f'  WARNING: no DNG found for stem "{stem}"')
            _dng_warned_missing.add(stem)
        _dng_crop_cache[stem] = None
        return None

    try:
        with rawpy.imread(dng_path) as raw:
            rgb = raw.postprocess(use_camera_wb=True, no_auto_bright=True, output_bps=8)
        h, w = rgb.shape[:2]
        side = max(1, int(min(h, w) * crop_fraction))
        cy, cx = h // 2, w // 2
        y0, x0 = max(0, cy - side // 2), max(0, cx - side // 2)
        cropped = rgb[y0:y0 + side, x0:x0 + side]
        img = cv2.resize(cropped, (size, size), interpolation=cv2.INTER_AREA)
        _dng_crop_cache[stem] = img
        return img
    except Exception as e:
        print(f'  WARNING: failed to load DNG {dng_path}: {e}')
        _dng_crop_cache[stem] = None
        return None


print(f'DNG central-crop helper ready (crop_fraction={DNG_CROP_FRACTION}, size={IMG_SIZE}).')

## 8. Training Function -- Balanced K-Fold CV, Weighted Sampling, RMSE/R2 Monitoring

In [ ]:
def train_kfold(data_list, run_name, n_folds=N_FOLDS, seed=SEED):
    """
    Balanced Stratified K-Fold CV on Pepper TIFF data.

    BALANCED ASSIGNMENT GUARANTEE
    ------------------------------
    For every (source, score_class) pair, samples are shuffled and assigned round-robin:
    fold_index = rank % n_folds. This guarantees every fold (validation set) contains
    images from every source present in data_list whenever that source has samples for
    a given class -- no data source is ever completely absent from validation.

    CLASS IMBALANCE
    ------------------------------
    Handled via (1) an inverse-frequency WeightedRandomSampler on the training fold and
    (2) target-value augmentation noise, in addition to the balanced fold assignment
    above. Monitoring / model selection is centered on RMSE (primary) and R2 (secondary).

    FINAL ARTIFACT
    ------------------------------
    Returns every fold's best-checkpoint state_dict ('fold_models') rather than a single
    cherry-picked fold, since the OOF metrics reported for this run are aggregated across
    all folds -- ensembling all fold models (see build_ensemble()/ensemble_predict() below)
    is what actually matches those reported metrics at inference time.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    rng = np.random.RandomState(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    all_scores = np.array([int(s) for _, s, _ in data_list])

    source_indices = defaultdict(list)
    for i, (_tif, _score, src) in enumerate(data_list):
        source_indices[src].append(i)
    sources = sorted(source_indices.keys())

    print(f'\n  {run_name}: n_folds={n_folds}, sources={sources}')

    sample_to_fold = {}
    print(f'\n  {"Source":<45}  {"Total":>6}  {"Folds w/ val":>13}  {"Val/fold (min-max)":>20}')
    print(f'  {"-"*45}  {"-"*6}  {"-"*13}  {"-"*20}')

    for src in sources:
        idxs = np.array(source_indices[src])
        src_scores = np.array([int(data_list[i][1]) for i in idxs])
        fold_assignment = np.full(len(idxs), -1, dtype=int)
        for cls in np.unique(src_scores):
            cls_local = np.where(src_scores == cls)[0]
            rng.shuffle(cls_local)
            for rank, local_i in enumerate(cls_local):
                fold_assignment[local_i] = rank % n_folds
        for local_i, fold_i in enumerate(fold_assignment):
            sample_to_fold[int(idxs[local_i])] = fold_i

        per_fold_counts = [int((fold_assignment == fi).sum()) for fi in range(n_folds)]
        nonempty = sum(1 for c in per_fold_counts if c > 0)
        print(f'  {src:<45}  {len(idxs):>6}  {nonempty:>4}/{n_folds}       '
              f'{min(per_fold_counts):>3} - {max(per_fold_counts):>3}')

    unassigned = set(range(len(data_list))) - set(sample_to_fold.keys())
    if unassigned:
        print(f'  WARNING: {len(unassigned)} unassigned samples -> fold 0')
        for gi in unassigned:
            sample_to_fold[gi] = 0

    fold_val_indices = [[] for _ in range(n_folds)]
    for global_idx, fold_i in sample_to_fold.items():
        fold_val_indices[fold_i].append(global_idx)

    active_folds = [fi for fi in range(n_folds) if len(fold_val_indices[fi]) > 0]
    print(f'\n  Active folds: {[f+1 for f in active_folds]}')
    print(f'  Val sizes:    {[len(fold_val_indices[fi]) for fi in active_folds]}')

    for fi in active_folds:
        val_sources_fi = set(data_list[i][2] for i in fold_val_indices[fi])
        missing = set(sources) - val_sources_fi
        if missing:
            print(f'  NOTE: fold {fi+1} has no validation samples from: {sorted(missing)}')

    oof_preds = np.full(len(data_list), np.nan)
    oof_targets = all_scores.astype(float)

    fold_metrics = []
    fold_histories = []
    fold_model_states = []

    for run_i, fold_idx in enumerate(active_folds):
        val_idx = np.array(fold_val_indices[fold_idx])
        val_set = set(val_idx.tolist())
        train_idx = np.array([i for i in range(len(data_list)) if i not in val_set])

        val_sources = set(data_list[i][2] for i in val_idx)
        print(f'\n  --- Fold {fold_idx+1}/{n_folds} (run {run_i+1}/{len(active_folds)}) ---')
        print(f'  Train={len(train_idx)} | Val={len(val_idx)} | Val sources={sorted(val_sources)}')

        train_data = [data_list[i] for i in train_idx]
        val_data = [data_list[i] for i in val_idx]

        train_ds = PepperDataset(train_data, IMG_SIZE, transform=train_transform, target_noise_std=0.15)
        val_ds = PepperDataset(val_data, IMG_SIZE, transform=val_transform, target_noise_std=0.0)

        train_sampler = make_weighted_sampler(train_data)
        train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=0)
        val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        model = RegressionCNN(3, IMG_SIZE, IMG_SIZE, DEPTH).to(device)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE)

        hist = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_rmse': [],
                'val_r2': [], 'val_spearman': [], 'lr': []}
        best_val_rmse = np.inf
        best_state = None
        no_improve = 0

        for epoch in range(NUM_EPOCHS):
            model.train()
            tl_sum = 0.0
            for imgs, tgts in train_dl:
                imgs, tgts = imgs.to(device), tgts.to(device)
                optimizer.zero_grad()
                loss = criterion(model(imgs), tgts)
                loss.backward()
                optimizer.step()
                tl_sum += loss.item() * imgs.size(0)
            tl = tl_sum / len(train_ds)

            model.eval()
            vl_sum = 0.0
            ep_preds, ep_tgts = [], []
            with torch.no_grad():
                for imgs, tgts in val_dl:
                    imgs, tgts = imgs.to(device), tgts.to(device)
                    out = model(imgs)
                    vl_sum += criterion(out, tgts).item() * imgs.size(0)
                    ep_preds.extend(out.cpu().numpy().flatten())
                    ep_tgts.extend(tgts.cpu().numpy().flatten())

            vl = vl_sum / len(val_ds)
            v_mae = mean_absolute_error(ep_tgts, ep_preds)
            v_rmse = np.sqrt(mean_squared_error(ep_tgts, ep_preds))
            v_r2 = r2_score(ep_tgts, ep_preds)
            sp = spearmanr(ep_tgts, ep_preds).correlation
            if np.isnan(sp):
                sp = 0.0
            cur_lr = optimizer.param_groups[0]['lr']
            scheduler.step(v_rmse)  # monitor RMSE (primary focus metric)

            hist['train_loss'].append(tl)
            hist['val_loss'].append(vl)
            hist['val_mae'].append(v_mae)
            hist['val_rmse'].append(v_rmse)
            hist['val_r2'].append(v_r2)
            hist['val_spearman'].append(sp)
            hist['lr'].append(cur_lr)

            if v_rmse < best_val_rmse:
                best_val_rmse = v_rmse
                best_state = copy.deepcopy(model.state_dict())
                no_improve = 0
            else:
                no_improve += 1

            if (epoch + 1) % 50 == 0 or epoch == 0:
                print(f'    Ep {epoch+1:4d} | RMSE:{v_rmse:.4f} R2:{v_r2:.3f} '
                      f'MAE:{v_mae:.3f} rho:{sp:.3f} LR:{cur_lr:.2e}')

            if no_improve >= EARLY_STOPPING_PATIENCE:
                print(f'    Early stop @ ep {epoch+1} (best RMSE={best_val_rmse:.4f})')
                break

        model.load_state_dict(best_state)
        model.eval()
        fp, ft = [], []
        with torch.no_grad():
            for imgs, tgts in val_dl:
                out = model(imgs.to(device))
                fp.extend(out.cpu().numpy().flatten())
                ft.extend(tgts.numpy().flatten())
        fp, ft = np.array(fp), np.array(ft)

        oof_preds[val_idx] = fp
        fold_model_states.append(copy.deepcopy(model.state_dict()))

        f_r2 = r2_score(ft, fp)
        f_sp = spearmanr(ft, fp).correlation if len(set(ft.tolist())) > 1 else 0.0
        if np.isnan(f_sp):
            f_sp = 0.0
        f_mae = mean_absolute_error(ft, fp)
        f_mse = mean_squared_error(ft, fp)
        f_rmse = np.sqrt(f_mse)

        fold_metrics.append({'R2': f_r2, 'RMSE': f_rmse, 'MSE': f_mse, 'MAE': f_mae,
                             'Spearman': f_sp, 'Epochs': len(hist['train_loss'])})
        fold_histories.append(hist)
        print(f'  Fold {fold_idx+1} | RMSE:{f_rmse:.4f} R2:{f_r2:.4f} '
              f'MAE:{f_mae:.3f} rho:{f_sp:.4f}')

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    valid_mask = ~np.isnan(oof_preds)
    oof_r2 = r2_score(oof_targets[valid_mask], oof_preds[valid_mask])
    oof_sp = spearmanr(oof_targets[valid_mask], oof_preds[valid_mask]).correlation
    if np.isnan(oof_sp):
        oof_sp = 0.0
    oof_mae = mean_absolute_error(oof_targets[valid_mask], oof_preds[valid_mask])
    oof_mse = mean_squared_error(oof_targets[valid_mask], oof_preds[valid_mask])
    oof_rmse = np.sqrt(oof_mse)

    mkeys = ['R2', 'RMSE', 'MSE', 'MAE', 'Spearman', 'Epochs']
    mean_m = {k: np.mean([fm[k] for fm in fold_metrics]) for k in mkeys}
    std_m = {k: np.std([fm[k] for fm in fold_metrics]) for k in mkeys}

    # NOTE: early stopping/checkpointing is tuned on the same fold it's scored on (standard
    # K-fold practice) -- this can be mildly optimistic for small folds, e.g. Groups B/C
    # where some score classes have very few samples per fold.
    print(f'\n  === {run_name} ({len(active_folds)}-fold CV) ===')
    print(f'  OOF   RMSE:{oof_rmse:.4f}  R2:{oof_r2:.4f}  rho:{oof_sp:.4f}  MAE:{oof_mae:.4f}  MSE:{oof_mse:.4f}')
    print(f'  Mean  RMSE:{mean_m["RMSE"]:.4f}+/-{std_m["RMSE"]:.4f}  R2:{mean_m["R2"]:.4f}+/-{std_m["R2"]:.4f}')

    return {
        'fold_models': fold_model_states,
        'fold_metrics': fold_metrics,
        'fold_histories': fold_histories,
        'mean_metrics': mean_m,
        'std_metrics': std_m,
        'oof_preds': oof_preds,
        'oof_targets': oof_targets,
        'oof_metrics': {'R2': oof_r2, 'RMSE': oof_rmse, 'MSE': oof_mse,
                        'MAE': oof_mae, 'Spearman': oof_sp},
        'n_folds': len(active_folds),
        'data_list': data_list,
        'sources': sources,
    }


def build_ensemble(fold_model_states):
    """Instantiate one RegressionCNN per fold's best-checkpoint state_dict, for
    ensembled inference (averaging predictions matches the OOF metrics reported
    above, unlike relying on a single cherry-picked fold)."""
    models = []
    for state in fold_model_states:
        m = RegressionCNN(3, IMG_SIZE, IMG_SIZE, DEPTH).to(device)
        m.load_state_dict(state)
        m.eval()
        models.append(m)
    return models


def ensemble_predict(models, x):
    """Average predictions across an ensemble of fold models. x: batch tensor on `device`."""
    with torch.no_grad():
        preds = torch.stack([m(x) for m in models], dim=0)
    return preds.mean(dim=0)


print('train_kfold() defined -- balanced per-source/class folds, weighted sampling, RMSE/R2-monitored.')
print('build_ensemble()/ensemble_predict() defined -- use the full 5-fold ensemble for future inference.')

## 9. Train All Groups x CC/NOCC (5-Fold CV)

In [ ]:
all_results = {gk: {} for gk in GROUP_KEYS}

for gk in GROUP_KEYS:
    for mode, GROUPS in MODE_GROUPS.items():
        print('=' * 70)
        print(f'TRAINING {gk} [{mode}] -- {GROUP_LABELS[gk]}')
        print('=' * 70)
        all_results[gk][mode] = train_kfold(GROUPS[gk], f'{gk}_{mode}', n_folds=N_FOLDS)
        print(f'{gk} [{mode}] done.\n')

print('All groups/modes trained.')

## 10. Visualization Helpers

In [ ]:
MODE_COLORS = {'CC': 'steelblue', 'NOCC': 'salmon'}


def group_metrics_table(gk):
    rows = []
    for mode in ['CC', 'NOCC']:
        res = all_results[gk][mode]
        om, mm, sm = res['oof_metrics'], res['mean_metrics'], res['std_metrics']
        rows.append({
            'Group': gk, 'Mode': mode, 'N': len(res['data_list']), 'Folds': res['n_folds'],
            'OOF_R2': f"{om['R2']:.4f}", 'OOF_RMSE': f"{om['RMSE']:.4f}",
            'OOF_MAE': f"{om['MAE']:.4f}", 'OOF_MSE': f"{om['MSE']:.4f}",
            'OOF_Spearman': f"{om['Spearman']:.4f}",
            'Mean_R2+/-Std': f"{mm['R2']:.3f}+/-{sm['R2']:.3f}",
            'Mean_RMSE+/-Std': f"{mm['RMSE']:.3f}+/-{sm['RMSE']:.3f}",
            'Mean_Epochs': f"{mm['Epochs']:.0f}",
        })
    return pd.DataFrame(rows)


def paired_cc_nocc_test(gk):
    """
    Wilcoxon signed-rank test on OOF |error| between CC and NOCC, matched by TIFF
    filename stem, for this group. Only runs if CC and NOCC are strictly 1:1 matched
    by stem -- otherwise it reports the mismatch and skips rather than forcing a
    pairing (see check_cc_nocc_parity() in Section 4 for the same check at load time).
    """
    res_cc = all_results[gk]['CC']
    res_nocc = all_results[gk]['NOCC']

    def stem_error_map(res):
        stems = [os.path.splitext(os.path.basename(t))[0] for t, _, _ in res['data_list']]
        errors = np.abs(res['oof_preds'] - res['oof_targets'])
        return dict(zip(stems, errors))

    cc_map = stem_error_map(res_cc)
    nocc_map = stem_error_map(res_nocc)
    common = sorted(set(cc_map) & set(nocc_map))

    print(f'\n  Paired CC vs NOCC significance test -- {gk}')
    if len(common) != len(cc_map) or len(common) != len(nocc_map):
        print(f'    SKIPPED: CC (N={len(cc_map)}) and NOCC (N={len(nocc_map)}) are not 1:1 '
              f'matched by stem ({len(common)} common) -- see check_cc_nocc_parity() in Section 4.')
        return None

    cc_err = np.array([cc_map[s] for s in common])
    nocc_err = np.array([nocc_map[s] for s in common])
    if np.allclose(cc_err - nocc_err, 0):
        print('    SKIPPED: CC and NOCC |error| identical for every matched sample.')
        return None

    stat, p_value = wilcoxon(cc_err, nocc_err)
    better = 'CC' if cc_err.mean() < nocc_err.mean() else 'NOCC'
    print(f'    Wilcoxon signed-rank (n={len(common)}): stat={stat:.4f}  p={p_value:.4f}  '
          f'-- {better} has lower mean |error| (CC={cc_err.mean():.4f}  NOCC={nocc_err.mean():.4f})')
    return {'n': len(common), 'statistic': stat, 'p_value': p_value,
            'cc_mean_abs_error': cc_err.mean(), 'nocc_mean_abs_error': nocc_err.mean()}


def plot_group_curves(gk):
    metrics_plot = [('val_rmse', 'Val RMSE (focus, lower=better)'),
                     ('val_r2', 'Val R2 (focus, higher=better)'),
                     ('val_loss', 'Val Loss/MSE (lower=better)'),
                     ('val_spearman', 'Val Spearman rho (higher=better)')]
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    for ax, (metric, title) in zip(axes, metrics_plot):
        for mode in ['CC', 'NOCC']:
            res = all_results[gk][mode]
            color = MODE_COLORS[mode]
            for fi, h in enumerate(res['fold_histories']):
                ep = range(1, len(h[metric]) + 1)
                lbl = mode if fi == 0 else None
                ax.plot(ep, h[metric], color=color, lw=1.0, alpha=0.5, label=lbl)
        ax.set_title(title, fontweight='bold', fontsize=10)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.suptitle(f'{gk} -- Training Curves CC vs NOCC ({GROUP_LABELS[gk]})', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_group_scatter(gk):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, mode in zip(axes, ['CC', 'NOCC']):
        res = all_results[gk][mode]
        preds, targets, om = res['oof_preds'], res['oof_targets'], res['oof_metrics']
        ax.scatter(targets, preds, alpha=0.7, s=70, c=MODE_COLORS[mode], edgecolors='black', linewidth=0.4)
        lo, hi = min(targets.min(), preds.min()), max(targets.max(), preds.max())
        ax.plot([lo, hi], [lo, hi], 'k--', lw=2, label='Perfect')
        ax.set_xlabel('True Score', fontweight='bold')
        ax.set_ylabel('Predicted Score', fontweight='bold')
        ax.set_title(f'{mode} -- R2={om["R2"]:.4f}  RMSE={om["RMSE"]:.4f}\n'
                     f'MAE={om["MAE"]:.3f}  rho={om["Spearman"]:.4f}', fontweight='bold', fontsize=10)
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.suptitle(f'{gk} -- OOF Predictions vs True ({GROUP_LABELS[gk]})', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_confusion_matrix_ax(data_list, oof_preds, title, ax):
    true_labels = np.array([int(s) for _, s, _ in data_list])
    pred_labels = np.clip(np.round(oof_preds).astype(int), int(true_labels.min()), int(true_labels.max()))
    labels = sorted(np.unique(np.concatenate([true_labels, pred_labels])))
    cm = confusion_matrix(true_labels, pred_labels, labels=labels)
    cm_norm = cm.astype(float)
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_norm = cm_norm / row_sums
    im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Predicted Score', fontsize=9)
    ax.set_ylabel('True Score', fontsize=9)
    ax.set_title(title, fontweight='bold', fontsize=10)
    for i in range(len(labels)):
        for j in range(len(labels)):
            val = cm[i, j]
            if val > 0:
                txt_color = 'white' if cm_norm[i, j] > 0.5 else 'black'
                ax.text(j, i, str(val), ha='center', va='center', fontsize=7, color=txt_color, fontweight='bold')
    return im


def plot_group_confusion(gk):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, mode in zip(axes, ['CC', 'NOCC']):
        res = all_results[gk][mode]
        im = plot_confusion_matrix_ax(res['data_list'], res['oof_preds'], f'{gk} -- {mode}', ax)
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.suptitle(f'{gk} -- Confusion Matrices (row-normalised)  {GROUP_LABELS[gk]}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    for mode in ['CC', 'NOCC']:
        res = all_results[gk][mode]
        true_l = np.array([int(s) for _, s, _ in res['data_list']])
        pred_l = np.clip(np.round(res['oof_preds']).astype(int), int(true_l.min()), int(true_l.max()))
        print(f'\n{"="*60}\nClassification Report -- {gk} [{mode}]\n{"="*60}')
        print(classification_report(true_l, pred_l, zero_division=0))


def draw_frieze_by_score(data_list, oof_preds, title, mode='pred', max_images_per_row=10):
    """
    One row per score (mode='pred' -> round(prediction), mode='true' -> true score).
    Images are the DNGs matching each TIFF stem, center-cropped to the produce skin.
    Rows are capped at `max_images_per_row`, evenly sub-sampled by rank if larger.
    """
    score_min = int(min(s for _, s, _ in data_list))
    score_max = int(max(s for _, s, _ in data_list))

    groups = {}
    for i, (tif, score, _src) in enumerate(data_list):
        pred = oof_preds[i]
        if np.isnan(pred):
            continue
        key = max(score_min, min(score_max, int(np.round(pred)))) if mode == 'pred' else int(score)
        groups.setdefault(key, []).append((tif, pred, int(score)))

    for k in groups:
        groups[k].sort(key=lambda x: x[1])
        total = len(groups[k])
        if total > max_images_per_row:
            step = total / max_images_per_row
            idxs = [int(i * step) for i in range(max_images_per_row)]
            groups[k] = [groups[k][idx] for idx in idxs]

    all_scores = sorted(groups.keys())
    max_per_row = max(len(groups[s]) for s in all_scores)
    n_rows = len(all_scores)

    label_w, img_w, img_h = 0.6, 0.9, 1.1
    fig, axes = plt.subplots(n_rows, max_per_row + 1,
                              figsize=(label_w + img_w * max_per_row, img_h * n_rows),
                              gridspec_kw={'width_ratios': [label_w / img_w] + [1] * max_per_row})
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    for row_i, g_score in enumerate(all_scores):
        items = groups[g_score]
        axes[row_i, 0].text(0.5, 0.5, str(g_score), fontsize=14, fontweight='bold',
                             ha='center', va='center', transform=axes[row_i, 0].transAxes)
        axes[row_i, 0].axis('off')
        for col_i in range(max_per_row):
            ax = axes[row_i, col_i + 1]
            if col_i < len(items):
                tif, pred, true_score = items[col_i]
                correct = (int(np.round(pred)) == true_score)
                color = 'green' if correct else 'red'
                img = get_cropped_dng(tif)
                if img is not None:
                    ax.imshow(img)
                    lbl = f'{pred:.1f}' if correct else f'{pred:.1f}\n(t:{true_score})'
                    ax.set_title(lbl, fontsize=5, color=color, pad=1)
                else:
                    ax.text(0.5, 0.5, 'N/A', ha='center', va='center', fontsize=6)
            ax.axis('off')

    mode_str = 'Predictions' if mode == 'pred' else 'Annotations'
    plt.suptitle(f'Frieze by {mode_str} (max {max_images_per_row}/row, DNG central crop) -- {title}',
                 fontsize=11, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


def show_group_comparison(gk):
    print('=' * 100)
    print(f'{gk} -- CC vs NOCC COMPARISON  ({GROUP_LABELS[gk]})')
    print('=' * 100)
    df = group_metrics_table(gk)
    print(df.to_string(index=False))
    paired_cc_nocc_test(gk)
    plot_group_curves(gk)
    plot_group_scatter(gk)
    plot_group_confusion(gk)
    for mode in ['CC', 'NOCC']:
        res = all_results[gk][mode]
        draw_frieze_by_score(res['data_list'], res['oof_preds'], f'{gk} [{mode}]',
                              mode='pred', max_images_per_row=10)
        draw_frieze_by_score(res['data_list'], res['oof_preds'], f'{gk} [{mode}]',
                              mode='true', max_images_per_row=10)


print('Visualization helpers defined.')

## 11. Group A -- CC vs NOCC Comparison

In [ ]:
show_group_comparison('Group_A')

## 12. Group B -- CC vs NOCC Comparison

In [ ]:
show_group_comparison('Group_B')

## 13. Group C -- CC vs NOCC Comparison

In [ ]:
show_group_comparison('Group_C')

## 14. Overall Summary -- All Groups & Modes

In [ ]:
summary_df = pd.concat([group_metrics_table(gk) for gk in GROUP_KEYS], ignore_index=True)
print('=' * 120)
print('OVERALL RESULTS SUMMARY -- All Groups x CC/NOCC (focus: R2, RMSE)')
print('=' * 120)
print(summary_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(GROUP_KEYS))
width = 0.35
for ax, metric_key, ylabel, better in [
    (axes[0], 'RMSE', 'RMSE', 'lower=better'),
    (axes[1], 'R2', 'R2', 'higher=better'),
]:
    for offset, mode in zip([-width / 2, width / 2], ['CC', 'NOCC']):
        means = [all_results[gk][mode]['mean_metrics'][metric_key] for gk in GROUP_KEYS]
        stds = [all_results[gk][mode]['std_metrics'][metric_key] for gk in GROUP_KEYS]
        bars = ax.bar(x + offset, means, width, yerr=stds, capsize=4,
                      label=mode, color=MODE_COLORS[mode], edgecolor='black')
        for bar, v in zip(bars, means):
            ax.annotate(f'{v:.3f}', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                        xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(GROUP_KEYS)
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} ({better}) -- mean +/- std', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Overall CC vs NOCC Comparison -- Focus Metrics (R2 / RMSE)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 15. Save Results

In [ ]:
out_root = os.path.join(PEPPER_ROOT, 'results_group_comparison_cc_nocc')
os.makedirs(out_root, exist_ok=True)
summary_df.to_csv(os.path.join(out_root, 'summary_groups_cc_nocc.csv'), index=False)

for gk in GROUP_KEYS:
    for mode in ['CC', 'NOCC']:
        res = all_results[gk][mode]
        gdir = os.path.join(out_root, gk, mode)
        os.makedirs(gdir, exist_ok=True)

        oof_df = pd.DataFrame({
            'tif_path': [t for t, _, _ in res['data_list']],
            'source': [src for _, _, src in res['data_list']],
            'true': res['oof_targets'],
            'predicted': res['oof_preds'],
            'abs_error': np.abs(res['oof_preds'] - res['oof_targets']),
        })
        oof_df.to_csv(os.path.join(gdir, 'oof_predictions.csv'), index=False)

        pd.DataFrame(res['fold_metrics']).to_csv(os.path.join(gdir, 'fold_metrics.csv'), index=False)

        for fi, hist in enumerate(res['fold_histories']):
            pd.DataFrame(hist).to_csv(os.path.join(gdir, f'history_fold{fi+1}.csv'), index=False)

        # Save the full 5-fold ensemble (list of state_dicts) rather than a single
        # cherry-picked fold, so the saved artifact matches the OOF metrics reported
        # above. Reload with: build_ensemble(torch.load('ensemble_fold_models.pt')).
        torch.save(res['fold_models'], os.path.join(gdir, 'ensemble_fold_models.pt'))
        print(f'  {gk}/{mode} saved -> {gdir}  ({len(res["fold_models"])}-model ensemble)')

print(f'\nAll results saved to: {out_root}')